# RSI Walk-Forward Optimizer
# Training: Pre-2023 | Testing: 2023+
# Optimization: RSI Threshold, ATR Filter, EMA21 Distance, Exits

In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
import random
import itertools
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
display(HTML("<style>.container { width:100% !important; }</style>"))

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'
TRAIN_END_DATE = '2023-01-01'


In [2]:
# 2. Data Loading & Core Indicator Calculations
def calc_rsi(series, period):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calc_atr(df, period):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    return true_range.rolling(window=period).mean()

all_data = []
files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
random.seed(42) 
random.shuffle(files)

loaded = 0
for file_path in files:
    if loaded >= 40: # Using 40 stocks to keep grid search fast
        break
    symbol = os.path.basename(file_path).replace('.csv', '')
    try:
        df = pd.read_csv(file_path).dropna(subset=['Close'])
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').reset_index(drop=True)
        
        if len(df) < 500: 
            continue
            
        latest_price = df['Close'].iloc[-1]
        if not (90 <= latest_price <= 600):
            continue
            
        df['EMA_89'] = df['Close'].ewm(span=89, adjust=False).mean()
        df['EMA_21'] = df['Close'].ewm(span=21, adjust=False).mean()
        df['RSI_3'] = calc_rsi(df['Close'], 3)
        
        atr_14 = calc_atr(df, 14)
        df['ATR_14_Pct'] = (atr_14 / df['Close']) * 100
        
        df['EMA21_Dist'] = ((df['Close'] - df['EMA_21']) / df['EMA_21']) * 100
        
        # We need data going back to ~2020 for training
        df = df.tail(1000).reset_index(drop=True)
        
        df['Symbol'] = symbol
        all_data.append(df)
        loaded += 1
    except Exception as e:
        pass

data = pd.concat(all_data, ignore_index=True)
print(f"Loaded and calculated {len(data['Symbol'].unique())} stocks.")


Loaded and calculated 40 stocks.


In [3]:
# 3. Vectorized Backtest Engine
def backtest_rsi(df, rsi_thresh, max_atr, max_ema_dist, take_profit, stop_loss, time_stop):
    trades = []
    in_trade = False
    entry_price = 0
    entry_date = None
    days_held = 0
    
    for i, row in df.iterrows():
        if pd.isna(row['EMA_89']):
            continue
            
        if in_trade:
            days_held += 1
            exit_triggered = False
            exit_reason = ""
            
            profit_pct = (row['Close'] - entry_price) / entry_price
            
            if profit_pct >= take_profit:
                exit_triggered = True
                exit_reason = "Take Profit"
            elif profit_pct <= stop_loss:
                exit_triggered = True
                exit_reason = "Stop Loss"
            elif days_held >= time_stop:
                exit_triggered = True
                exit_reason = "Time Stop"
            
            if exit_triggered:
                trades.append({
                    'Symbol': row['Symbol'],
                    'Entry_Date': entry_date,
                    'Exit_Date': row['Date'],
                    'Return': profit_pct,
                    'Reason': exit_reason
                })
                in_trade = False
                days_held = 0
                
        if not in_trade:
            # RSI Setup: Crosses above threshold from below
            # Prev RSI must be < threshold
            # Current RSI >= threshold
            prev_rsi = df['RSI_3'].iloc[i-1] if i > 0 else np.nan
            
            if pd.notna(prev_rsi) and prev_rsi < rsi_thresh and row['RSI_3'] >= rsi_thresh:
                # Core Rule
                if row['Close'] > row['EMA_89']:
                    # Avoid Rules
                    if row['ATR_14_Pct'] <= max_atr and row['EMA21_Dist'] <= max_ema_dist:
                        in_trade = True
                        entry_price = row['Close']
                        entry_date = row['Date']
                        
    return trades


In [4]:
# 4. Grid Search Optimizer
# Grid Space (Shrunk for speed)
RSI_THRESH = [20, 30]
MAX_ATR = [6, 999] # 999 = No limit
MAX_EMA_DIST = [-1, 3]
TAKE_PROFIT = [0.05, 0.08]
STOP_LOSS = [-0.03, -0.05]
TIME_STOP = [5, 8]

train_data = data[data['Date'] < TRAIN_END_DATE].copy()
test_data = data[data['Date'] >= TRAIN_END_DATE].copy()

leaderboard = []

total_combinations = len(RSI_THRESH) * len(MAX_ATR) * len(MAX_EMA_DIST) * len(TAKE_PROFIT) * len(STOP_LOSS) * len(TIME_STOP)
print(f"Starting Walk-Forward Optimization. Grid size: {total_combinations} combinations...")

run_count = 0
for rsi, atr, ema, tp, sl, ts in itertools.product(RSI_THRESH, MAX_ATR, MAX_EMA_DIST, TAKE_PROFIT, STOP_LOSS, TIME_STOP):
    run_count += 1
    if run_count % 100 == 0:
        print(f"Processed {run_count} / {total_combinations}...")
        
    # Phase 1: Train
    train_trades = []
    for sym, grp in train_data.groupby('Symbol'):
        train_trades.extend(backtest_rsi(grp.reset_index(drop=True), rsi, atr, ema, tp, sl, ts))
        
    train_df = pd.DataFrame(train_trades)
    
    # Must have enough trades and > 45% win rate in training to even bother testing
    if len(train_df) < 10: continue
    train_win_rate = (train_df['Return'] > 0).mean()
    if train_win_rate < 0.45: continue
        
    # Phase 2: Test on Unseen Data (2023+)
    test_trades = []
    for sym, grp in test_data.groupby('Symbol'):
        test_trades.extend(backtest_rsi(grp.reset_index(drop=True), rsi, atr, ema, tp, sl, ts))
        
    test_df = pd.DataFrame(test_trades)
    
    if len(test_df) < 5: continue
        
    test_win_rate = (test_df['Return'] > 0).mean()
    test_avg_ret = test_df['Return'].mean()
    
    if test_avg_ret <= 0: continue
        
    # Phase 3: Calculate Fitness
    test_df = test_df.sort_values('Exit_Date').reset_index(drop=True)
    equity = 10000 * (1 + test_df['Return']).cumprod()
    drawdown = ((equity / equity.cummax()) - 1).min()
    drawdown = abs(drawdown) if drawdown < 0 else 0.01
    
    fitness = (test_win_rate * test_avg_ret) / drawdown
    
    params_str = f"RSI_X:{rsi} | Max_ATR:{atr}% | Max_EMA21_Dist:{ema}% | TP:+{int(tp*100)}% | SL:{int(sl*100)}% | TS:{ts}d"
    
    leaderboard.append({
        'Params': params_str,
        'Test_Trades': len(test_df),
        'Test_WinRate': test_win_rate * 100,
        'Test_AvgReturn': test_avg_ret * 100,
        'Test_Drawdown': drawdown * 100,
        'Fitness': fitness
    })

print("Optimization Complete.")


Starting Walk-Forward Optimization. Grid size: 64 combinations...


Optimization Complete.


In [5]:
# 5. Leaderboard Showdown
if len(leaderboard) > 0:
    leaderboard_df = pd.DataFrame(leaderboard).sort_values('Fitness', ascending=False).reset_index(drop=True)
    
    display(HTML("<h3>Leaderboard Showdown (Tested on Unseen 2023+ Data)</h3>"))
    
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: lightgreen' if v else '' for v in is_max]
        
    styled_df = leaderboard_df.head(15).style.apply(highlight_max, subset=['Fitness', 'Test_WinRate', 'Test_AvgReturn']).format({
        'Test_WinRate': "{:.2f}%",
        'Test_AvgReturn': "{:.2f}%",
        'Test_Drawdown': "{:.2f}%",
        'Fitness': "{:.4f}"
    })
    
    display(styled_df)
    
    print("\n🏆 CHAMPION STRATEGY:")
    print("=============================================")
    print(leaderboard_df.iloc[0]['Params'])
else:
    print("No strategies survived the Walk-Forward Optimization (No profitable setups found on unseen data).")


,Params,Test_Trades,Test_WinRate,Test_AvgReturn,Test_Drawdown,Fitness
0,RSI_X:30 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-5% | TS:5d,239,51.46%,0.71%,37.11%,0.0098
1,RSI_X:30 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-5% | TS:8d,226,49.56%,0.73%,42.15%,0.0086
2,RSI_X:30 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-3% | TS:8d,239,44.77%,0.72%,43.50%,0.0074
3,RSI_X:30 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+5% | SL:-5% | TS:5d,239,52.72%,0.54%,38.63%,0.0073
4,RSI_X:20 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-5% | TS:8d,250,49.60%,0.68%,49.66%,0.0067
5,RSI_X:20 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-3% | TS:8d,264,45.45%,0.72%,49.06%,0.0066
6,RSI_X:30 | Max_ATR:6% | Max_EMA21_Dist:-1% | TP:+8% | SL:-5% | TS:5d,219,49.77%,0.50%,37.48%,0.0066
7,RSI_X:30 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-3% | TS:5d,251,47.41%,0.60%,43.25%,0.0066
8,RSI_X:20 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-5% | TS:5d,269,51.67%,0.51%,40.76%,0.0065
9,RSI_X:20 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+5% | SL:-5% | TS:5d,269,53.16%,0.46%,38.45%,0.0064



🏆 CHAMPION STRATEGY:
RSI_X:30 | Max_ATR:999% | Max_EMA21_Dist:-1% | TP:+8% | SL:-5% | TS:5d
